## Setup

In [ ]:
import json
import pandas as pd
import re
from pathlib import Path
import requests


pd.set_option('display.max_colwidth', None)

In [ ]:
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

DATA = ROOT / "data" / "all_article_urls.json"


df = pd.read_json(DATA)
df.rename(columns={0 : 'URL'},inplace=True)
df.head()

In [ ]:
# Target the standard APS DOI prefix (10.1103) within the URL string
# Since the list of APS URLs already exists, DOI can be extracted for making bibliogrpay API calls
def extract_doi(url):
    if not isinstance(url, str):
        return None
    m = re.search(r"10\.1103/[^\s]+", url)
    return m.group(0) if m else None


# Test extraction on a sample row
sample_url = df.iloc[5]["URL"]
sample_doi = extract_doi(sample_url)
print(f"Sample URL: {sample_url}")
print(f"Extracted DOI: {sample_doi}")

## Checking various APIs

### SemanticScholar API

In [ ]:
# Testing a single paper metadata retrieval via DOI
if sample_doi:
    ss_url = f"https://api.semanticscholar.org/graph/v1/paper/DOI:{sample_doi}"
    params = {"fields": "title,abstract,year"}

    r = requests.get(ss_url, params=params)
    if r.status_code == 200:
        ss_data = r.json()
        print("--- Semantic Scholar Match ---")
        print(f"Title: {ss_data.get('title')}")
        print(f"Abstract Snippet: {str(ss_data.get('abstract'))[:100]}...")
    else:
        print(f"Semantic Scholar API failed with status code: {r.status_code}")

### InspireHEP API

In [ ]:
if sample_doi:
    inspire_url = f"https://inspirehep.net/api/literature?q=doi:{sample_doi}"
    inspire_data = requests.get(inspire_url).json()
    hits = inspire_data.get("hits", {}).get("hits", [])
    print(f"InspireHEP total record hits for DOI: {len(hits)}")
    print(f"Title for DOI {sample_doi} : {inspire_data.get('title')}")

### OpenAlex API

In [ ]:
openalex_url = "https://api.openalex.org/works"
openalex_params = {
        "filter": "primary_location.source.id:S188941785",  # Physical Review B
        "select": "title,abstract_inverted_index,concepts",
        "per_page": 200
    }
openalex_response = requests.get(openalex_url, params=openalex_params)

if openalex_response.status_code == 200: 
    print("Succesfull connection API")
    openalex_data=openalex_response.json()
    print(openalex_data.get("results",[]))

### APS Harvest API (Best solution - has phySH tag metadata)

In [ ]:
headers = {"Accept": "application/vnd.tesseract.article+json"}

harvestAPS_params = {
    "from": "2025-08-01",
    "until": "2025-08-31", 
    "journals": "PRB",
    "date": "published",
    "per_page": 100
}

harvestAPS_url="https://harvest.aps.org/v2/journals/articles"

response = requests.get(harvestAPS_url, headers=headers, params = harvestAPS_params)
print(response.headers.get("link"))
print(f"Bibliography fields pulled: {response.json().get('data',None)[0].keys()}")

In [ ]:
#Manually checking pagination mechanics
response = requests.get("https://harvest.aps.org/v2/journals/articles?date=published&from=2025-01-01&page=21&per_page=100&until=2025-02-01",headers=headers)
response.links.get("next")
datatmp=response.json()

### Preparing API list for batch processing

In [ ]:
doi_list=df['URL'].apply(extract_doi).dropna().to_list()
ss_url_list=[f"https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}" for doi in doi_list]

print(f"Generated {len(ss_url_list)} target API URLs.")
print("Sample URLs:", ss_url_list[0:4])